# Fine-Tune Whisper untuk Al-Quran Juz 30

**Qari referensi:** Mahmoud Khalil Al-Husary (128kbps)  
**Dataset:** 564 ayat Juz 30 (Surah 78–114)  
**Base model:** `naazimsnh02/whisper-large-v3-turbo-ar-quran`  
**Estimasi waktu:** ~45 menit di Colab T4

### Cara Pakai
1. Pastikan Runtime → Change runtime type → **GPU (T4)**
2. Isi `HF_TOKEN` dan `HF_USERNAME` di Cell 2
3. **Runtime → Run All**
4. Tinggalkan — selesai otomatis, model di-push ke HuggingFace


In [ ]:
# ─────────────────────────────────────────
# Cell 1: Install semua library yang dibutuhkan
# ─────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate jiwer \
    huggingface_hub pydub librosa soundfile tensorboard
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
print('✅ Library terinstall')

In [ ]:
# ─────────────────────────────────────────
# Cell 2: KONFIGURASI — ISI BAGIAN INI
# ─────────────────────────────────────────
from huggingface_hub import login

# Dapatkan token di: https://huggingface.co/settings/tokens
# Pilih role: write
HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxx"  # ← GANTI INI
HF_USERNAME = "your-username"            # ← GANTI INI (username HF kamu)
MODEL_NAME  = f"{HF_USERNAME}/whisper-quran-juz30-husary"

BASE_MODEL  = "naazimsnh02/whisper-large-v3-turbo-ar-quran"
SURAH_START = 78   # Juz 30 mulai surah 78
SURAH_END   = 114  # sampai surah 114
QARI        = "Husary_128kbps"
EVERYAYAH   = "https://everyayah.com/data"

login(token=HF_TOKEN)
print(f'✅ Login HuggingFace sebagai: {HF_USERNAME}')
print(f'✅ Model akan disimpan di: {MODEL_NAME}')

In [ ]:
# ─────────────────────────────────────────
# Cell 3: Download audio Al-Husary Juz 30
# ─────────────────────────────────────────
import os, requests, time
from pathlib import Path

AUDIO_DIR = Path("/content/audio_husary")
AUDIO_DIR.mkdir(exist_ok=True)

# Data ayat per surah Juz 30
JUZ30_VERSES = {
    78:40, 79:46, 80:42, 81:29, 82:19, 83:36, 84:25, 85:22,
    86:17, 87:19, 88:26, 89:30, 90:20, 91:15, 92:21, 93:11,
    94:8,  95:8,  96:19, 97:5,  98:8,  99:8,  100:11, 101:11,
    102:8, 103:3, 104:9, 105:5, 106:4, 107:7, 108:3, 109:6,
    110:3, 111:5, 112:4, 113:5, 114:6
}

downloaded, skipped, failed = 0, 0, 0

for surah, total_verses in JUZ30_VERSES.items():
    for verse in range(1, total_verses + 1):
        filename = f"{surah:03d}{verse:03d}.mp3"
        filepath = AUDIO_DIR / filename

        if filepath.exists():
            skipped += 1
            continue

        url = f"{EVERYAYAH}/{QARI}/{filename}"
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                filepath.write_bytes(r.content)
                downloaded += 1
            else:
                print(f'  ⚠ {filename}: HTTP {r.status_code}')
                failed += 1
        except Exception as e:
            print(f'  ✗ {filename}: {e}')
            failed += 1
        time.sleep(0.05)  # jangan terlalu cepat

    print(f'Surah {surah} ✓ ({total_verses} ayat)')

total = downloaded + skipped
print(f'\n✅ Selesai: {total} file audio ({downloaded} baru, {skipped} sudah ada, {failed} gagal)')

In [ ]:
# ─────────────────────────────────────────
# Cell 4: Buat teks Arab per ayat dari verses.json
# ─────────────────────────────────────────
import json

# Download verses.json dari repo kamu atau paste langsung
# Opsi 1: Upload manual ke Colab (/content/verses.json)
# Opsi 2: Ambil dari HuggingFace dataset (di bawah)

# Kita pakai Quran API sebagai sumber teks Uthmani
VERSES_FILE = Path("/content/verses_juz30.json")

if not VERSES_FILE.exists():
    print("Mengambil teks ayat dari API Quran.com...")
    verses_data = {}
    for surah in JUZ30_VERSES.keys():
        url = f"https://api.qurancdn.com/api/qdc/verses/by_chapter/{surah}?words=false&per_page=300&fields=text_uthmani"
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            verses_data[str(surah)] = [
                {
                    "verse_number": v["verse_number"],
                    "text_uthmani": v["text_uthmani"]
                }
                for v in data["verses"]
            ]
            print(f'  Surah {surah}: {len(verses_data[str(surah)])} ayat')
        else:
            print(f'  ⚠ Surah {surah}: gagal ({r.status_code})')
        time.sleep(0.2)
    VERSES_FILE.write_text(json.dumps(verses_data, ensure_ascii=False, indent=2))
    print(f'✅ Teks ayat disimpan ke {VERSES_FILE}')
else:
    verses_data = json.loads(VERSES_FILE.read_text())
    print(f'✅ Teks ayat dimuat dari cache')

# Buat mapping: filename → teks
audio_text_pairs = []
for surah, total_verses in JUZ30_VERSES.items():
    surah_verses = verses_data.get(str(surah), [])
    for v in surah_verses:
        filename = f"{surah:03d}{v['verse_number']:03d}.mp3"
        filepath = AUDIO_DIR / filename
        if filepath.exists():
            # Bersihkan teks: strip nomor ayat di awal/akhir
            text = v['text_uthmani'].strip()
            # Hapus karakter angka Arab di akhir (١٢٣ dll)
            import re
            text = re.sub(r'[\u0660-\u0669\u06F0-\u06F9]+$', '', text).strip()
            audio_text_pairs.append({
                "audio_path": str(filepath),
                "text": text,
                "surah": surah,
                "verse": v['verse_number']
            })

print(f'✅ Total pasangan audio-teks: {len(audio_text_pairs)}')
print(f'\nContoh:')
for p in audio_text_pairs[:3]:
    print(f"  {p['surah']}:{p['verse']} → {p['text'][:50]}...")

In [ ]:
# ─────────────────────────────────────────
# Cell 5: Preprocessing audio → WAV 16kHz
# ─────────────────────────────────────────
import librosa
import soundfile as sf
import numpy as np

WAV_DIR = Path("/content/audio_wav")
WAV_DIR.mkdir(exist_ok=True)

converted = 0
for pair in audio_text_pairs:
    mp3_path = Path(pair['audio_path'])
    wav_path = WAV_DIR / (mp3_path.stem + ".wav")

    if not wav_path.exists():
        try:
            audio, sr = librosa.load(mp3_path, sr=16000, mono=True)
            sf.write(wav_path, audio, 16000)
            converted += 1
        except Exception as e:
            print(f'  ✗ {mp3_path.name}: {e}')
            continue

    pair['wav_path'] = str(wav_path)

# Filter hanya yang punya wav_path
audio_text_pairs = [p for p in audio_text_pairs if 'wav_path' in p]

print(f'✅ Audio dikonversi: {converted} file baru')
print(f'✅ Total dataset siap: {len(audio_text_pairs)} ayat')

In [ ]:
# ─────────────────────────────────────────
# Cell 6: Buat HuggingFace Dataset
# ─────────────────────────────────────────
from datasets import Dataset, Audio
import random

# Shuffle dulu
random.seed(42)
random.shuffle(audio_text_pairs)

# Split 90% train, 10% validation
split_idx = int(len(audio_text_pairs) * 0.9)
train_data = audio_text_pairs[:split_idx]
val_data   = audio_text_pairs[split_idx:]

def make_dataset(pairs):
    return Dataset.from_dict({
        "audio": [p['wav_path'] for p in pairs],
        "sentence": [p['text'] for p in pairs],
    }).cast_column("audio", Audio(sampling_rate=16000))

train_dataset = make_dataset(train_data)
val_dataset   = make_dataset(val_data)

print(f'✅ Train: {len(train_dataset)} sampel')
print(f'✅ Validation: {len(val_dataset)} sampel')
print(f'\nContoh train[0]:')
print(f"  text: {train_dataset[0]['sentence']}")

In [ ]:
# ─────────────────────────────────────────
# Cell 7: Load model & processor
# ─────────────────────────────────────────
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device: {device}')
if device == "cuda":
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'\nMemuat model {BASE_MODEL}...')
processor = WhisperProcessor.from_pretrained(BASE_MODEL)
model     = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

# Wajib untuk fp16 + gradient_checkpointing agar tidak error saat training
model.config.use_cache = False
model.enable_input_require_grads()

# Force Arabic transcription
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print(f'✅ Model dimuat ({sum(p.numel() for p in model.parameters())/1e6:.0f}M parameter)')

In [ ]:
# ─────────────────────────────────────────
# Cell 8: Preprocessing features (mel spectrogram + tokenize)
# ─────────────────────────────────────────
from dataclasses import dataclass
from typing import Any, Dict, List, Union

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("Preprocessing train dataset...")
train_dataset = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names,
    num_proc=2
)

print("Preprocessing validation dataset...")
val_dataset = val_dataset.map(
    prepare_dataset,
    remove_columns=val_dataset.column_names,
    num_proc=2
)

print(f'✅ Preprocessing selesai')

# Data collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
# ─────────────────────────────────────────
# Cell 9: Metric evaluasi (WER)
# ─────────────────────────────────────────
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print('✅ Metric WER siap')

In [ ]:
# ─────────────────────────────────────────
# Cell 10: Konfigurasi Training
# ─────────────────────────────────────────
import gc
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

# Bersihkan GPU memory sebelum training
gc.collect()
torch.cuda.empty_cache()
print(f'GPU allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

training_args = Seq2SeqTrainingArguments(
    output_dir=f"/content/{MODEL_NAME.split('/')[-1]}",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,     # efektif batch size = 16
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=5,
    gradient_checkpointing=False,      # dimatikan — fix konflik fp16 + gradient_checkpointing
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    report_to=["tensorboard"],
    push_to_hub=True,
    hub_model_id=MODEL_NAME,
    hub_strategy="every_save",
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print('✅ Trainer siap')
print(f'   Epochs       : 5 (max, early stopping aktif)')
print(f'   Batch size   : 16 (efektif, 2×8 gradient accumulation)')
print(f'   Learning rate: 1e-5')
print(f'   Checkpoint   : setiap epoch → push ke HF')

In [ ]:
# ─────────────────────────────────────────
# Cell 11: CEK BASELINE (opsional, skip jika mau langsung training)
# ─────────────────────────────────────────
# Lewati cell ini, langsung ke Cell 12 untuk mulai training
print('⏭ Cell 11 di-skip — langsung ke Cell 12 untuk training')

In [ ]:
# ─────────────────────────────────────────
# Cell 12: MULAI TRAINING
# ─────────────────────────────────────────
# Ini bagian yang bisa ditinggal.
# Progress tampil di bawah cell ini.
# Model di-push ke HuggingFace setiap epoch sebagai backup.

print('🚀 Training dimulai...')
print(f'   Model akan di-push ke: https://huggingface.co/{MODEL_NAME}')
print()

trainer.train()

print('\n✅ Training selesai!')

In [ ]:
# ─────────────────────────────────────────
# Cell 13: Simpan model final ke HuggingFace
# ─────────────────────────────────────────
trainer.push_to_hub(
    commit_message="Fine-tuned on Quran Juz 30 - Al-Husary 128kbps"
)
processor.push_to_hub(MODEL_NAME)

print(f'✅ Model final tersimpan di:')
print(f'   https://huggingface.co/{MODEL_NAME}')

In [ ]:
# ─────────────────────────────────────────
# Cell 14: EVALUASI HASIL (setelah fine-tune)
# ─────────────────────────────────────────
from transformers import pipeline as hf_pipeline

pipe_final = hf_pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    torch_dtype=torch.float16,
    device=device,
)

print('=== HASIL SETELAH FINE-TUNE ===')
for p in audio_text_pairs[:5]:
    result = pipe_final(p['wav_path'], generate_kwargs={"language": "arabic"})
    print(f"Expected  : {p['text']}")
    print(f"Predicted : {result['text'].strip()}")
    print()

In [ ]:
# ─────────────────────────────────────────
# Cell 15: Cara pakai di backend lokal
# ─────────────────────────────────────────
print("""Untuk pakai model baru di tilawah_service.py:

Ganti baris ini:
  model = AutoModelForSpeechSeq2Seq.from_pretrained(
      'naazimsnh02/whisper-large-v3-turbo-ar-quran',
      ...
  )
  _processor = AutoProcessor.from_pretrained('naazimsnh02/whisper-large-v3-turbo-ar-quran')

Dengan:
  model = AutoModelForSpeechSeq2Seq.from_pretrained(
      '{MODEL_NAME}',
      ...
  )
  _processor = AutoProcessor.from_pretrained('{MODEL_NAME}')

Restart backend → model download otomatis (~800MB, sekali saja)
""".format(MODEL_NAME=MODEL_NAME))